# Lab6: 批量文本提示分割
## Batch Text-Prompt Segmentation with Labeled Composite

---

读取 `config/batch_prompts.txt` 中的构件描述，对每张图片用所有提示词做纯文本分割，
生成带 ID + 置信度标注的复合图。

In [1]:
# ========== 环境准备 ==========
import os, sys, csv, json, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from pathlib import Path
%matplotlib inline

print("[✅] 环境就绪")

[✅] 环境就绪


In [2]:
# ========== 路径配置 ==========
_cwd = Path.cwd().resolve()
if (_cwd / "sam3").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "sam3").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find repo root from {_cwd}")

CKPT_PATH = REPO_ROOT / "sam3.pt"
INPUT_DIR = REPO_ROOT / "Inputs" / "RawImages"
PROMPTS_PATH = REPO_ROOT / "config" / "batch_prompts.txt"
LAB6_OUTPUT = REPO_ROOT / "Outputs" / "Lab6_output"

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"权重未找到: {CKPT_PATH}")
if not PROMPTS_PATH.exists():
    raise FileNotFoundError(f"提示词未找到: {PROMPTS_PATH}")

# 读取提示词
prompts = []
with open(PROMPTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            prompts.append(line)

print(f"提示词: {len(prompts)} 个")
for i, p in enumerate(prompts):
    print(f"  {i+1}. {p}")

# 图片列表（支持多种后缀）
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}
image_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if Path(f).suffix.lower() in IMAGE_EXTENSIONS
])

LAB6_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"图片: {len(image_files)} 张")
print(f"输出: {LAB6_OUTPUT}")
print(f"权重: {CKPT_PATH.name}")

提示词: 27 个
  1. swallowtail ridge
  2. horse-back gable
  3. porcelain inlay
  4. dougong bracket
  5. red brick wall
  6. stone door frame
  7. waterwheel frieze
  8. roof ridge sculpture
  9. carved beam
  10. painted ceiling
  11. wooden window lattice
  12. stone plinth
  13. eave tile
  14. cylindrical and flat roof tiles
  15. drip tile
  16. main ridge & hip ridge
  17. rolling-shed roof
  18. stone-and-brick mosaic wall
  19. white stone plinth
  20. stone pillar
  21. wall painting
  22. openwork stone lattice window
  23. stone sill
  24. wooden door leaf
  25. couplet plaques
  26. stone-paved courtyard
  27. stone slabs
图片: 13 张
输出: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab6_output
权重: sam3.pt


In [3]:
# ========== 加载 SAM 3 模型 ==========
print("[⏳] 加载 SAM 3 模型...")
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    设备: {device}")

model = build_sam3_image_model(
    checkpoint_path=str(CKPT_PATH),
    load_from_HF=False,
    device=device,
    eval_mode=True,
)
processor = Sam3Processor(model, device=device)
pm = sum(p.numel() for p in model.parameters())/1e6
print(f"[✅] SAM 3 | 参数: {pm:.0f}M")

[⏳] 加载 SAM 3 模型...
    设备: cuda
[✅] SAM 3 | 参数: 841M


In [4]:
# ========== 核心处理函数 ==========

VIS_AREA_THRESHOLD = 0.005  # 最小掩码比例
_cmap = plt.cm.tab20


def process_single(img_path, img_id):
    """对一张图跑所有提示词，返回结果列表。"""
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    img_area = w * h
    print(f"\n[📷] {img_id} ({w}x{h})")

    all_masks = []  # (mask_np, score, prompt_idx)
    for pi, prompt in enumerate(prompts):
        try:
            state = processor.set_image(img)
            state = processor.set_text_prompt(prompt=prompt, state=state)
            masks = state["masks"]
            scores = state["scores"]
            count = 0
            for i in range(len(masks)):
                score = scores[i].item()
                mask = masks[i].squeeze().cpu().numpy()
                if mask.sum() / img_area < VIS_AREA_THRESHOLD:
                    continue
                all_masks.append((mask, score, pi))
                count += 1
            if count:
                print(f"    {prompt}: {count}")
        except Exception as e:
            print(f"    {prompt}: ERROR - {e}")

    print(f"    总计: {len(all_masks)} 个掩码")
    return all_masks, img


def save_labeled_composite(all_masks, img, img_id, out_dir):
    """生成标注复合图：掩码叠加 + 每个掩码显示 ID 和分数。"""
    img_np = np.array(img)
    h, w = img_np.shape[:2]
    overlay = np.zeros((h, w, 4), dtype=np.float32)

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(img_np)

    legend_patches = []
    used_prompts = {}
    for idx, (mask, score, pi) in enumerate(all_masks):
        color = _cmap(pi % 20)
        overlay[mask] = list(color[:3]) + [0.5]
        if pi not in used_prompts:
            used_prompts[pi] = color
            legend_patches.append(mpatches.Patch(color=color, label=prompts[pi]))
        # 标注
        ys, xs = np.where(mask)
        if len(xs) > 0:
            cx, cy = np.mean(xs), np.mean(ys)
            ax.text(cx, cy, f"#{idx}:{score:.2f}",
                    color='white', fontsize=6, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.08', facecolor='black', alpha=0.6))

    ax.imshow(overlay)
    ax.set_title(f"{img_id} | {len(all_masks)} masks")
    ax.axis("off")
    ax.legend(handles=legend_patches, loc="upper right",
              fontsize=7, framealpha=0.7, ncol=2)
    plt.tight_layout()
    fig.savefig(out_dir / f"{img_id}_labeled.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"    复合图已保存: {out_dir / f'{img_id}_labeled.png'}")


print("[✅] 处理函数已定义")

[✅] 处理函数已定义


In [5]:
# ========== 执行批量分割 ==========
all_results = []  # 每张图的结果

for img_file in image_files:
    img_id = Path(img_file).stem
    img_path = INPUT_DIR / img_file

    try:
        masks_data, img = process_single(img_path, img_id)
        if not masks_data:
            print(f"[⚠] {img_id}: 无有效掩码")
            all_results.append({"img_id": img_id, "masks": [], "path": img_path})
            continue

        # 创建输出目录
        out_dir = LAB6_OUTPUT / img_id
        out_dir.mkdir(parents=True, exist_ok=True)

        # 保存每个掩码 PNG
        meta_list = []
        for idx, (mask, score, pi) in enumerate(masks_data):
            mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
            fname = f"{idx:04d}_{prompts[pi].replace(' ','_')}_{score:.3f}.png"
            mask_pil.save(out_dir / fname)
            meta_list.append({
                "id": idx,
                "prompt": prompts[pi],
                "score": score,
                "file": fname,
            })

        # 保存元数据
        with open(out_dir / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(meta_list, f, indent=2, ensure_ascii=False)

        # 生成标注复合图
        save_labeled_composite(masks_data, img, img_id, out_dir)

        all_results.append({
            "img_id": img_id,
            "masks": masks_data,
            "path": img_path,
        })
    except Exception as e:
        print(f"\n[❌] {img_id} 处理失败: {e}")
        import traceback; traceback.print_exc()

print(f"\n[✅] 处理完成 | {len(all_results)} 张图片")


[📷] 00_RawImage (1080x690)
    red brick wall: 5
    总计: 5 个掩码
    复合图已保存: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab6_output\00_RawImage\00_RawImage_labeled.png

[📷] 01_RawImage (622x417)
    red brick wall: 5
    总计: 5 个掩码
    复合图已保存: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab6_output\01_RawImage\01_RawImage_labeled.png

[📷] 02_RawImage (1618x1080)
    red brick wall: 5
    stone door frame: 1
    roof ridge sculpture: 2
    stone-and-brick mosaic wall: 7
    stone pillar: 4
    openwork stone lattice window: 2
    stone-paved courtyard: 2
    总计: 23 个掩码
    复合图已保存: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab6_output\02_RawImage\02_RawImage_labeled.png

[📷] 03_RawImage (2545x1697)
    red brick wall: 2
    总计: 2 个掩码
    复合图已保存: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab6_output\03_RawImage\03_RawImage_labeled.png

[📷] 04_RawImage (2581x1721)
    red brick wall: 9
    roof ridge sculpture: 1
    总计: 10 个掩码
    复合图已保存: C:\Users\Legion\Desktop\WIE\SAM3_Labs\O

In [6]:
# ========== 汇总报告 ==========
print("=" * 100)
print("Lab6: 批量文本提示分割 - 汇总报告")
print("=" * 100)

header_cols = ["图片", "尺寸", "总数"] + prompts
col_width = max(16, max(len(p) for p in prompts) + 2)
col_w = [10, 14, 6] + [col_width] * len(prompts)

header = " | ".join(h.ljust(w)[:w] for h, w in zip(header_cols, col_w))
sep = "-" * len(header)
print(header)
print(sep)

total_all_masks = 0
prompt_totals = {p: 0 for p in prompts}

for r in all_results:
    img_id = r["img_id"]
    masks_data = r["masks"]
    w, h = Image.open(r["path"]).size if r["masks"] else (0, 0)
    n_masks = len(masks_data)
    total_all_masks += n_masks

    # 每个提示词的计数
    prompt_counts = {p: 0 for p in prompts}
    for _, _, pi in masks_data:
        prompt_counts[prompts[pi]] = prompt_counts.get(prompts[pi], 0) + 1
    for p in prompts:
        prompt_totals[p] += prompt_counts.get(p, 0)

    row = [img_id[:10], f"{w}x{h}", str(n_masks)]
    for p in prompts:
        row.append(str(prompt_counts.get(p, 0)))
    print(" | ".join(v.ljust(w)[:w] for v, w in zip(row, col_w)))

print(sep)
row = ["总计", "", str(total_all_masks)]
for p in prompts:
    row.append(str(prompt_totals[p]))
print(" | ".join(v.ljust(w)[:w] for v, w in zip(row, col_w)))
print()

# 输出到文件
summary_path = LAB6_OUTPUT / "summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(f"Lab6 汇总 | {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"提示词: {len(prompts)} | 图片: {len(all_results)} | 总掩码: {total_all_masks}\n")
    f.write("=" * 80 + "\n")
    for r in all_results:
        f.write(f"{r['img_id']}: {len(r['masks'])} masks\n")
print(f"[✅] 汇总: {summary_path}")

Lab6: 批量文本提示分割 - 汇总报告
图片         | 尺寸             | 总数     | swallowtail ridge                 | horse-back gable                  | porcelain inlay                   | dougong bracket                   | red brick wall                    | stone door frame                  | waterwheel frieze                 | roof ridge sculpture              | carved beam                       | painted ceiling                   | wooden window lattice             | stone plinth                      | eave tile                         | cylindrical and flat roof tiles   | drip tile                         | main ridge & hip ridge            | rolling-shed roof                 | stone-and-brick mosaic wall       | white stone plinth                | stone pillar                      | wall painting                     | openwork stone lattice window     | stone sill                        | wooden door leaf                  | couplet plaques                   | stone-paved courtyard             | sto

### 说明

- 提示词定义在 `config/batch_prompts.txt`，可随时修改
- 每张图片的输出在 `Outputs/Lab6_output/{img_id}/` 下
- `{img_id}_labeled.png` 是带 ID 和分数的标注复合图
- 纯文本分割速度较快但精度不如框+文本，适合快速全量扫描